# S3 Search: the engine end to end, and a quick benchmark against OpenSearch

> **DEVELOPMENT ONLY. NEVER THE SCORE OF RECORD.** Every number this notebook prints comes from a small, quick,
> single-client run on whatever Colab machine you got. The score of record is `referee/colab_referee.ipynb`
> (`python -m referee full`), run by the founder. The rigorous load test is `colab/fair_bench.py` (open loop, pinned
> client CPUs, GPU and CPU sampling). This notebook is the guided tour, not either of those.

What it does, top to bottom (everything goes through the HTTP API of `docs/INTERFACES.md` section 7):

1. **Setup**: clone the repository at `REF`, install what is missing, detect the device, load the data **only through `referee/loader.py`**.
2. **Start the engine as a service**: `python -m serving` in the background.
3. **Index through the API**: `PUT /indexes/{name}` with a mapping, then `GET` the index and its `_mapping`; `_bulk` NDJSON.
4. **Query through the API**: the tuned BM25F request, facets, filters, multi-select `post_filter`, pagination (`from` and `search_after`), `_msearch`, `explain`, and a small dense and hybrid example.
5. **Live updates**: patch, delete, upsert, `_refresh`, `versions`, `_reindex`, `_rollback`.
6. **Benchmark against OpenSearch 2.19.1** on the same catalogue and the same query texts: indexing time, nDCG@10 and recall (**fit-split judgments only**), batch-1 latency, and a short concurrent run. Predictions are written down in a cell **before** the benchmark.

Runtime: a GPU runtime (A100 or T4) is what the engine is for; without one it runs on CPU, slower. `DATASET = "esci-sample"`
(about 49,000 products, minutes) is the default; `"esci-us"` is the full 1.2 million products (a long run, and 10 GB or more of RAM for OpenSearch).

## 1. Setup

The repository is public, so no secret is needed. If it is ever made private, add a Colab **secret named `GITHUB_TOKEN`**
(read access to `hanialshater/s3-search`) and switch on "Notebook access"; the token is never printed.

In [ ]:
# @title Settings
REF = "main"  # @param {type:"string"}
DATASET = "esci-sample"  # @param ["esci-sample", "esci-us"]
RUN_OPENSEARCH = True  # @param {type:"boolean"}
N_QUALITY = 500  # @param {type:"integer"}
N_LATENCY = 300  # @param {type:"integer"}
SECONDS_PER_LEVEL = 10  # @param {type:"integer"}
CONCURRENCY = [1, 4, 16, 64]
DEPTH = 10            # hits per request in the latency and throughput runs (page 1); quality uses 100
OS_VERSION = "2.19.1"
OS_HEAP_MAX_GB = 8
DENSE_PRODUCTS = 20000  # products in the small dense/hybrid example index
LIVE_PRODUCTS = 5000    # products in the live-updates example index

import json, os
_overrides = os.environ.get("S3_DEMO_OVERRIDES")   # for running this notebook unattended (tests); empty in Colab
if _overrides:
    globals().update(json.loads(_overrides))
    print("overrides:", _overrides)

In [ ]:
# @title Setup: clone, install what is missing, device, data
import atexit, copy, importlib.util, math, shutil, signal, socket, subprocess, sys, tarfile, threading, time
import urllib.error, urllib.parse, urllib.request
from http.client import HTTPConnection, HTTPException
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np
import pandas as pd

IN_COLAB = importlib.util.find_spec("google.colab") is not None
REPO_URL = "https://github.com/hanialshater/s3-search"
WORK = Path("/content/s3-demo") if IN_COLAB else Path(os.environ.get("S3_DEMO_WORK", "/tmp/s3-demo"))
WORK.mkdir(parents=True, exist_ok=True)

def say(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

if IN_COLAB:
    REPO = Path("/content/s3-search")
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        pass
    url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL
    def git(*args, check=True):
        r = subprocess.run(["git", "-C", str(REPO), *args], capture_output=True, text=True)
        if check and r.returncode:
            msg = (r.stderr or r.stdout)
            raise RuntimeError("git " + " ".join(a for a in args if "x-access-token" not in a) + " failed: "
                               + (msg.replace(token, "***") if token else msg))
        return r
    if not REPO.exists():
        r = subprocess.run(["git", "clone", "--quiet", url, str(REPO)], capture_output=True, text=True)
        if r.returncode:
            raise RuntimeError("git clone failed. If the repository is private, add a Colab secret GITHUB_TOKEN with read "
                               "access to it and allow notebook access. " + (r.stderr.replace(token, "***") if token else r.stderr))
        git("remote", "set-url", "origin", REPO_URL)          # the token does not stay in .git/config
    if git("fetch", "--quiet", url, REF, check=False).returncode == 0:   # a branch, a tag or a full sha
        git("checkout", "--quiet", "FETCH_HEAD")
    else:                                                      # a short sha
        git("fetch", "--quiet", url, "+refs/heads/*:refs/remotes/origin/*")
        git("checkout", "--quiet", REF)
    # Only what this runtime is missing: requirements.txt is satisfied by Colab's numpy and pandas. requirements-gpu.txt is
    # NOT installed: as referee/colab_referee.ipynb notes, its cu12 wheels would overwrite Colab's own torch and RAPIDS.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")], check=True)
else:
    REPO = Path(os.environ["S3_DEMO_REPO"]) if os.environ.get("S3_DEMO_REPO") else next(
        p for p in [Path.cwd(), *Path.cwd().parents] if (p / "referee" / "loader.py").exists())
os.chdir(REPO)
sys.path.insert(0, str(REPO))

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else None
except Exception:
    DEVICE, GPU_NAME = "cpu", None
COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip() or "unknown"
say(f"repo {REPO} at {COMMIT} (REF {REF if IN_COLAB else 'local checkout'}); device {DEVICE}"
    + (f" ({GPU_NAME})" if GPU_NAME else "") + f"; {os.cpu_count()} CPUs; dataset {DATASET}")

from referee import loader, metrics        # the only door to ESCI (the floor's data rule)
if DATASET == "esci-us" or not loader.available(DATASET):
    say("fetching the ESCI files (about 1.6 GB, once)")
    loader.fetch()
ds = loader.load_dataset(DATASET)
say(f"{len(ds.catalogue):,} products; {ds.fit['query_id'].nunique():,} fit-split queries with judgments; "
    f"{len(ds.test_queries):,} test query texts (no judgments: used for latency only)")

In [ ]:
# @title Helpers: HTTP, waiting for servers, stopping them
class ApiError(RuntimeError):
    pass

_opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))      # localhost: never through a proxy

def http(method, url, body=None, *, raw=None, ctype="application/json", timeout=900, ok=(200, 201)):
    # One JSON call. Raises ApiError (with the server's error body) unless the status is in `ok`.
    data = raw if raw is not None else (None if body is None else json.dumps(body).encode())
    req = urllib.request.Request(url, data=data, method=method, headers={"Content-Type": ctype})
    try:
        with _opener.open(req, timeout=timeout) as r:
            status, payload = r.status, r.read()
    except urllib.error.HTTPError as e:
        status, payload = e.code, e.read()
    try:
        parsed = json.loads(payload) if payload else None
    except ValueError:
        parsed = payload.decode(errors="replace")
    if ok and status not in ok:
        raise ApiError(f"{method} {url} -> HTTP {status}: {json.dumps(parsed)[:600]}")
    return parsed

def ndjson(*lines):
    return ("\n".join(json.dumps(x) for x in lines) + "\n").encode()

def free_port():
    s = socket.socket(); s.bind(("127.0.0.1", 0)); p = s.getsockname()[1]; s.close(); return p

def tail(path, n=40):
    try:
        return "\n".join(Path(path).read_text(errors="replace").splitlines()[-n:])
    except OSError:
        return "(no log)"

def wait_until(check, what, seconds, proc=None, log=None):
    # Poll `check` until it returns something truthy. If the process dies or the time runs out, print its log and raise.
    t0, last = time.time(), None
    while time.time() - t0 < seconds:
        if proc is not None and proc.poll() is not None:
            raise RuntimeError(f"{what}: the process exited with {proc.returncode}. Log tail:\n{tail(log)}")
        try:
            if check():
                return
        except Exception as e:           # not up yet
            last = e
        time.sleep(0.5)
    raise RuntimeError(f"{what}: not ready after {seconds:.0f} s (last error: {last}). Log tail:\n{tail(log)}")

PIDFILE = WORK / "pids.json"
PROCS = {}

def stop_proc(proc):
    if proc is not None and proc.poll() is None:
        try:
            os.killpg(proc.pid, signal.SIGTERM)
            proc.wait(60)
        except subprocess.TimeoutExpired:
            os.killpg(proc.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass

def stop_all():
    for name, p in list(PROCS.items()):
        stop_proc(p)
        say(f"stopped {name}")
    PROCS.clear()
    PIDFILE.unlink(missing_ok=True)

def spawn(name, cmd, log, env=None):
    p = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=open(log, "w"), stderr=subprocess.STDOUT, start_new_session=True)
    PROCS[name] = p
    PIDFILE.write_text(json.dumps({n: q.pid for n, q in PROCS.items()}))
    return p

def kill_stale():
    # Servers left by an earlier run of this notebook (a kernel restart does not stop them): stop them first.
    if PIDFILE.exists():
        for name, pid in json.loads(PIDFILE.read_text()).items():
            try:
                os.killpg(pid, signal.SIGTERM); say(f"stopped a stale {name} (pid {pid})")
            except (ProcessLookupError, PermissionError):
                pass
        time.sleep(2)
    PIDFILE.unlink(missing_ok=True)

atexit.register(stop_all)

## 2. Start the engine as a service

`python -m serving` is the HTTP front over the engine (`serving/README.md`). It starts with no index: `/healthz` answers at once, and
`/readyz` turns 200 when an index is loaded (it is 503 until then, which is correct). The query cache is **off**
(`S3_SERVING_CACHE=0`, as in `colab/fair_bench.py`), so the benchmark measures the engine and not the cache.

In [ ]:
# @title Start serving
kill_stale()
ENGINE_DIR = WORK / "serving"
shutil.rmtree(ENGINE_DIR, ignore_errors=True)
ENGINE_PORT = free_port()
ENGINE = f"http://127.0.0.1:{ENGINE_PORT}"
ENGINE_LOG = WORK / "serving.log"
env = {**os.environ, "S3_SEARCH_DEVICE": DEVICE, "S3_SERVING_CACHE": "0"}
proc = spawn("engine", [sys.executable, "-m", "serving", "--data-dir", str(ENGINE_DIR), "--device", DEVICE,
                        "--port", str(ENGINE_PORT)], ENGINE_LOG, env)
wait_until(lambda: http("GET", ENGINE + "/healthz")["status"] == "ok", "serving /healthz", 300, proc, ENGINE_LOG)
say(f"serving is up at {ENGINE} on {DEVICE} (log {ENGINE_LOG})")
print("GET /healthz ->", http("GET", ENGINE + "/healthz"))
print("GET /readyz  ->", http("GET", ENGINE + "/readyz", ok=(200, 503)), "(not ready: no index yet)")

def api(method, path, body=None, **kw):
    return http(method, ENGINE + path, body, **kw)

## 3. Index through the API

**The catalogue.** ESCI products through `referee/loader.py`, plus three synthetic columns (`price`, `stock`, `category`, the same recipe
as `tools/stack.py`, seed 7) so that filters, numeric facets and in-place patches have something to work on. ESCI has none of them.
OpenSearch gets the very same columns later.

**The mapping** is `loader.ESCI_MAPPING` (the five text fields with the engine's `s3_english` analyzer, brand and colour with `.keyword` sub-fields),
plus the three new fields, with `price` and `stock` declared **hot** (patchable in place), and the tuned per-field BM25F parameters
`referee/requests/bm25f-tuned.bm25.json` under `_meta.engine.bm25`.

`PUT /indexes/{name}` takes `{"mapping", "source": {"path"}}`: intake, build a segment, load it. Here the source is a Parquet file on the same machine.

In [ ]:
# @title The catalogue and the mapping
def with_synthetic_columns(cat):
    cat = cat.copy()
    rng = np.random.default_rng(7)                                   # as tools/stack.py
    cat["price"] = np.round(rng.uniform(1, 500, len(cat)), 2)
    cat["stock"] = (rng.integers(0, 4, len(cat)) * rng.integers(0, 40, len(cat))).astype(np.int32)
    cat["category"] = rng.choice(["apparel", "home", "electronics", "toys", "sports", "beauty"], len(cat))
    return cat

EXTRA_PROPS = {"price": {"type": "float"}, "stock": {"type": "integer"}, "category": {"type": "keyword"}}
cat = with_synthetic_columns(ds.catalogue)

ENGINE_BM25 = json.loads((REPO / "referee/requests/bm25f-tuned.bm25.json").read_text())
engine_mapping = copy.deepcopy(ds.mapping)
engine_mapping["properties"].update(EXTRA_PROPS)
engine_mapping["_meta"]["engine"]["hot"] = ["price", "stock"]
engine_mapping["_meta"]["engine"]["bm25"] = ENGINE_BM25

print(cat.head(3)[["product_id", "product_title", "product_brand", "price", "stock", "category"]].to_string(max_colwidth=50))
print(json.dumps(engine_mapping["_meta"], indent=1)[:700])

In [ ]:
# @title PUT /indexes/esci: the whole catalogue
INDEX = "esci"
SRC = WORK / "catalogue.parquet"
cat.to_parquet(SRC, index=False)
t0 = time.time()
made = api("PUT", f"/indexes/{INDEX}", {"mapping": engine_mapping, "source": {"path": str(SRC)}}, timeout=7200)
ENGINE_INDEX_S = time.time() - t0
say(f"PUT /indexes/{INDEX}: {len(cat):,} products indexed and loaded in {ENGINE_INDEX_S:.1f} s -> {made}")
print("GET /readyz  ->", api("GET", "/readyz"))
print("GET /indexes/esci ->", json.dumps(api("GET", f"/indexes/{INDEX}")))
fields = api("GET", f"/indexes/{INDEX}/_mapping")[INDEX]["s3_fields"]
print("\nGET /indexes/esci/_mapping -> s3_fields (what each field can do):")
print(pd.DataFrame(fields).T[["type", "searchable", "filterable", "facetable", "sortable", "hot"]].to_string())

### `_bulk`: OpenSearch NDJSON

A second, small index, `esci-live`, is created the same way from the first few thousand products, and then **gets new documents through `_bulk`**.
It is also where the live-updates section works, so the big benchmark index is never touched by the demo.

In [ ]:
# @title _bulk NDJSON into a small index
LIVE = "esci-live"
live_cat = cat.iloc[:LIVE_PRODUCTS].reset_index(drop=True)
live_cat.to_parquet(WORK / "live.parquet", index=False)
api("PUT", f"/indexes/{LIVE}", {"mapping": engine_mapping, "source": {"path": str(WORK / "live.parquet")}}, timeout=3600)
print("created", LIVE, api("GET", f"/indexes/{LIVE}")["n_docs"], "docs")

new_docs = [{"product_id": f"DEMO-{i:04d}", "product_title": f"Demo product number {i} for the bulk example",
             "product_brand": "S3 Demo", "product_color": "Black", "product_bullet_point": "A document added by _bulk.",
             "product_description": "Added through the bulk API.", "price": 10.0 + i, "stock": i, "category": "home"}
            for i in range(1, 6)]
lines = []
for d in new_docs:
    lines += [{"index": {"_id": d["product_id"]}}, d]
t0 = time.time()
res = api("POST", f"/indexes/{LIVE}/_bulk?refresh=true", raw=ndjson(*lines), ctype="application/x-ndjson")
print(f"_bulk of {len(new_docs)} documents with refresh=true: {(time.time() - t0) * 1000:.0f} ms, errors={res['errors']}, "
      f"results={[list(i.values())[0]['result'] for i in res['items']]}")
r = api("POST", f"/indexes/{LIVE}/_search", {"query": {"match": {"product_title": "bulk example"}}, "size": 3, "_source": ["product_title", "price"]})
print("searching the new documents:", r["hits"]["total"], "hits;", [(h["_id"], h["_source"]["price"]) for h in r["hits"]["hits"]])

## 4. Query through the API

The request is a subset of the OpenSearch query DSL (`docs/INTERFACES.md` section 5). The lexical request is the **tuned BM25F** one the referee uses,
`referee/requests/bm25f-tuned.json` (a `multi_match` over the five fields with tuned boosts, scored as BM25F).

The three demo queries are the fit-split queries with the most judgments (deterministic), so they surely have results.

In [ ]:
# @title The tuned request, three readable queries
ENGINE_TMPL = json.loads((REPO / "referee/requests/bm25f-tuned.json").read_text())
print("request template:", json.dumps(ENGINE_TMPL))

def engine_body(q, size=DEPTH, **extra):
    b = json.loads(json.dumps(ENGINE_TMPL).replace("{{query}}", json.dumps(q)[1:-1]))
    b["size"] = size
    b.update(extra)
    return b

def esearch(body, index=INDEX):
    return api("POST", f"/indexes/{index}/_search", body)

def show(resp, n=5, width=72):
    print(f"  {resp['hits']['total']:,} matches; took_ms {resp.get('took_ms', {}).get('total')}")
    for i, h in enumerate(resp["hits"]["hits"][:n], 1):
        s = h.get("_source") or {}
        extra = "".join(f"  {k}={s[k]}" for k in ("price", "stock", "category") if s.get(k) is not None)
        print(f"  {i}. {h['_id']:<12} {h['_score']:8.3f}  {str(s.get('product_brand') or '')[:14]:<14} {str(s.get('product_title', ''))[:width]}{extra}")

SRC_FIELDS = ["product_title", "product_brand", "price", "stock", "category"]
counts = ds.fit.groupby("query_id").size().sort_values(ascending=False, kind="stable")
DEMO_QUERIES = [ds.fit.loc[ds.fit["query_id"] == qid, "query"].iloc[0] for qid in counts.index[:3]]
for q in DEMO_QUERIES:
    print(f"\n> {q!r}")
    show(esearch(engine_body(q, 5, _source=SRC_FIELDS)))
Q = DEMO_QUERIES[0]

In [ ]:
# @title Facets (aggregations) over the whole match set, a filter, and multi-select with post_filter
print(f"> {Q!r} with facets (counts are over ALL matches, not only the page):")
r = esearch(engine_body(Q, 3, _source=SRC_FIELDS, aggs={
    "brands": {"terms": {"field": "product_brand.keyword", "size": 5}},
    "categories": {"terms": {"field": "category", "size": 6}},
    "price_buckets": {"range": {"field": "price", "ranges": [{"to": 50}, {"from": 50, "to": 200}, {"from": 200}]}},
    "price": {"stats": {"field": "price"}}}))
show(r, 3)
agg = r["aggregations"]
print("  brands:    ", [(b["key"], b["doc_count"]) for b in agg["brands"]["buckets"]])
print("  categories:", [(b["key"], b["doc_count"]) for b in agg["categories"]["buckets"]])
print("  price:     ", [(b["key"], b["doc_count"]) for b in agg["price_buckets"]["buckets"]], "stats:", {k: round(v, 2) for k, v in agg["price"].items()})

print(f"\n> the same query, filtered: category = electronics and price <= 100 (a bool filter does not change scores):")
filt = engine_body(Q, 3, _source=SRC_FIELDS)
filt["query"] = {"bool": {"must": [filt["query"]], "filter": [{"term": {"category": "electronics"}}, {"range": {"price": {"lte": 100}}}]}}
show(esearch(filt), 3)

top_brand = agg["brands"]["buckets"][0]["key"]
print(f"\n> multi-select: post_filter brand = {top_brand!r}. The hits narrow, the facet counts do not:")
ms = engine_body(Q, 3, _source=SRC_FIELDS, aggs={"categories": {"terms": {"field": "category", "size": 6}}},
                 post_filter={"term": {"product_brand.keyword": top_brand}})
r2 = esearch(ms)
show(r2, 3)
print("  categories still:", [(b["key"], b["doc_count"]) for b in r2["aggregations"]["categories"]["buckets"]])

In [ ]:
# @title Pagination: from/size and search_after give the same page 2
p1 = esearch(engine_body(Q, 5, _source=["product_title"]))
last_sort = p1["hits"]["hits"][-1]["sort"]               # the last hit's sort values, verbatim
p2_after = esearch(engine_body(Q, 5, _source=["product_title"], search_after=last_sort))
p2_from = esearch(engine_body(Q, 5, _source=["product_title"], **{"from": 5}))
ids = lambda r: [h["_id"] for h in r["hits"]["hits"]]
print("page 1:", ids(p1)); print("page 2 by search_after:", ids(p2_after)); print("page 2 by from=5:      ", ids(p2_from))
assert ids(p2_after) == ids(p2_from), "the two ways of paging disagree"
print("same page 2 both ways: True")

In [ ]:
# @title _msearch: several requests in one call
batch = [engine_body(q, 3, _source=["product_title"]) for q in DEMO_QUERIES]
t0 = time.time(); rs = api("POST", f"/indexes/{INDEX}/_msearch", batch); dt = (time.time() - t0) * 1000
print(f"_msearch of {len(batch)} requests: {dt:.1f} ms")
for q, r in zip(DEMO_QUERIES, rs):
    print(f"  {q!r}: {r['hits']['total']:,} matches; top: {r['hits']['hits'][0]['_source']['product_title'][:70]}")

In [ ]:
# @title explain: why did this product rank where it did?
from tools.common import http_search
from tools.explain import explain_product, render_text
search_fn = http_search(ENGINE, INDEX)
third = esearch(engine_body(Q, 3))["hits"]["hits"][2]["_id"]
e = explain_product(search_fn, engine_body(Q), third)        # tools/explain.py: finds the hit, checks every sum against its children
print("\n".join(render_text(e).splitlines()[:26]))
print(f"... ({e['n_checked']} sums checked against their children, {e['n_failed']} failed; root equals the hit's _score: {e['root_matches_score']})")

### Dense and hybrid (a small example)

`esci-dense` is the first `DENSE_PRODUCTS` products with a `knn_vector` field `embedding`. Its vectors come from the engine's built-in
**`tfidf-svd-64`** encoder (TF-IDF reduced with SVD, fitted on the catalogue at build), a development-grade encoder that needs no model download.
It is **not** the learned encoder the GPU pipeline is meant for, so read the dense numbers as a demonstration of the API, not of quality.
`hybrid` runs the lexical and the `neural` query and fuses their ranked lists with reciprocal rank fusion (`rank.rrf`).

In [ ]:
# @title neural and hybrid (RRF)
try:
    DENSE = "esci-dense"
    dm = copy.deepcopy(engine_mapping)
    dm["properties"]["embedding"] = {"type": "knn_vector", "dimension": 64,
                                     "method": {"name": "hnsw", "space_type": "cosinesimil", "engine": "lucene"}}
    dm["_meta"]["engine"]["embeddings"] = {"embedding": {"model": "tfidf-svd-64",
                                                          "inputs": ["product_title", "product_brand", "product_color"]}}
    cat.iloc[:DENSE_PRODUCTS].to_parquet(WORK / "dense.parquet", index=False)
    t0 = time.time()
    api("PUT", f"/indexes/{DENSE}", {"mapping": dm, "source": {"path": str(WORK / "dense.parquet")}}, timeout=3600)
    say(f"{DENSE}: {min(DENSE_PRODUCTS, len(cat)):,} products with 64-d vectors in {time.time() - t0:.1f} s")
    text = DEMO_QUERIES[1]
    lex = engine_body(text)["query"]
    neural = {"neural": {"embedding": {"query_text": text, "k": 100}}}
    for name, body in [("lexical (BM25F)", {"query": lex}),
                       ("neural (tfidf-svd-64)", {"query": neural}),
                       ("hybrid (RRF of both)", {"query": {"hybrid": {"queries": [lex, neural]}},
                                                 "rank": {"rrf": {"rank_constant": 60, "window_size": 100}}})]:
        r = esearch({**body, "size": 3, "_source": ["product_title"]}, DENSE)
        print(f"\n{name} for {text!r}:")
        for h in r["hits"]["hits"]:
            print(f"   {h['_score']:7.3f}  {h['_source']['product_title'][:80]}")
except Exception as ex:
    print("dense/hybrid example skipped:", ex)

## 5. Live updates

On `esci-live`. Semantics (INTERFACES section 7, "Refresh"): a **patch** of a hot field and a **delete** are visible at once; an **upsert** of a
new document is visible at the next **refresh** (every `refresh_interval`, 500 ms by default, or when asked with `_refresh`). To make the wait visible we turn the background refresh off
for a moment.

In [ ]:
# @title patch, delete, upsert and refresh
def lsearch(text, size=3, **kw):
    return esearch({"query": {"match": {"product_title": text}}, "size": size, "_source": ["product_title", "price", "stock"], **kw}, LIVE)
def bulk(*lines, refresh="false"):
    return api("POST", f"/indexes/{LIVE}/_bulk?refresh={refresh}", raw=ndjson(*lines), ctype="application/x-ndjson")

LQ = DEMO_QUERIES[0].split()[0]
r = lsearch(LQ); print(f"> {LQ!r} before:"); show(r)
top = r["hits"]["hits"][0]["_id"]

print(f"\n1. patch {top}: price 1.0, stock 0  (hot fields: in place, visible at once)")
print("  ", bulk({"update": {"_id": top}}, {"doc": {"price": 1.0, "stock": 0}})["items"][0])
show(lsearch(LQ), 1)

print(f"\n2. delete {top}")
print("  ", bulk({"delete": {"_id": top}})["items"][0])
show(lsearch(LQ), 1)

print("\n3. upsert a new document with refresh turned off")
api("PUT", f"/indexes/{LIVE}/_settings", {"index": {"refresh_interval": "-1"}})
newdoc = {"product_id": "DEMO-9999", "product_title": "QZDEMO unique upsert example", "product_brand": "S3 Demo", "product_color": "Red",
          "product_bullet_point": "x", "product_description": "x", "price": 42.0, "stock": 7, "category": "toys"}
print("  ", bulk({"index": {"_id": "DEMO-9999"}}, newdoc)["items"][0])
print("   visible before _refresh:", lsearch("qzdemo")["hits"]["total"], "hits;  pending upserts:", api("GET", f"/indexes/{LIVE}")["pending_upserts"])
print("   POST _refresh ->", api("POST", f"/indexes/{LIVE}/_refresh", {}))
r = lsearch("qzdemo"); print("   visible after _refresh: ", r["hits"]["total"], "hits:", [h["_id"] for h in r["hits"]["hits"]])

In [ ]:
# @title versions, _reindex and _rollback
v0 = api("GET", f"/indexes/{LIVE}/versions")
print("versions:", [(x["segment_id"], x["live"]) for x in v0])
first_segment = v0[0]["segment_id"]
t0 = time.time()
print("POST _reindex ->", api("POST", f"/indexes/{LIVE}/_reindex", {}, timeout=3600), f"({time.time() - t0:.1f} s: a new segment built from the store, then swapped in)")
v1 = api("GET", f"/indexes/{LIVE}/versions")
print("versions:", [(x["segment_id"], x["live"]) for x in v1])
print(f"after the reindex  : qzdemo -> {lsearch('qzdemo')['hits']['total']} hit(s); deleted {top} -> {lsearch(LQ, 50)['hits']['total']} matches for {LQ!r}")

print(f"\nPOST _rollback to {first_segment} (the segment the PUT built):")
print("  ", api("POST", f"/indexes/{LIVE}/_rollback", {"segment_id": first_segment}))
print("   live:", [(x["segment_id"], x["live"]) for x in api("GET", f"/indexes/{LIVE}/versions")])
info = api("GET", f"/indexes/{LIVE}")
print(f"   the change log is replayed on the older segment: the delete still holds ({top} in results: "
      f"{top in [h['_id'] for h in lsearch(LQ, 50)['hits']['hits']]}); pending upserts: {info['pending_upserts']}")
api("POST", f"/indexes/{LIVE}/_refresh", {})
print("   after _refresh: qzdemo ->", lsearch("qzdemo")["hits"]["total"], "hit(s)")
api("PUT", f"/indexes/{LIVE}/_settings", {"index": {"refresh_interval": "500ms"}})
print("refresh_interval back to", api("GET", f"/indexes/{LIVE}/_settings"))

## 6. Benchmark against OpenSearch 2.19.1

**Predictions, written before any run.** They are committed with this notebook, and the results are judged against them, wrong ones included.
Sources: `baseline/README.md` (the founder's Colab prototype, 2026-11-05, ESCI-US, A100: tuned lexical nDCG@10 **0.304** for the engine's BM25F against **0.270** for
OpenSearch `cross_fields`, on the **test** split; page 1, batch 1, **full pipeline with the ranker**, p50 / p99 **20.6 / 33.8 ms** against **268 / 379 ms**; pipeline throughput under a 100 ms p99 **275 QPS**);
`docs/BOARD.md` S6-13 (an A100 in-process profile, 2026-11-13: **lexical** batch-1 p50 about **7 ms**, depth 2,000, before HTTP); `docs/BOARD.md` S6-7 (the first HTTP load test of
2026-11-13, depth 2,000: both engines sustained about **50 QPS**; the server has been reworked since). No recorded number in the repository covers indexing time, or either engine on `esci-sample`;
those predictions are labelled **unsourced**, with a belief instead.

| # | Prediction (on the machine you run it on) | Source | Wrong if |
|---|---|---|---|
| P1 | Quality, fit split: the engine's tuned BM25F has **higher nDCG@10 than OpenSearch tuned `cross_fields`**, by **0.01 to 0.06** (the prototype's gap on test is 0.034). The engine's parameters were tuned on this split, so its fit-split score is, if anything, a little generous. | baseline/README.md | The engine is not higher, or the gap is above 0.08 |
| P2 | Quality, absolute, `esci-us`: both within **0.05** of the prototype's test numbers (0.304 and 0.270). On `esci-sample` the catalogue is only judged products, so both are **much higher**: no prior, recorded here for the first time. | baseline/README.md (esci-us only) | `esci-us`: either outside 0.05 of its prototype number |
| P3 | Batch-1 latency over HTTP, **`esci-us`, GPU**: engine **p50 under 25 ms** and **p99 under 60 ms** (the in-process lexical 7 ms, plus the 2 ms batching window, HTTP and JSON); **OpenSearch p50 above the engine's**. | BOARD S6-13; baseline/README.md (ordering) | Engine p50 over 25 ms or p99 over 60 ms; or OpenSearch p50 at or below the engine's |
| P4 | Batch-1 latency, **`esci-sample`** (about 49,000 products): the corpus is small, so OpenSearch is fast too and the gap **narrows or reverses**. Belief 0.4 that the engine still has the lower p50. | unsourced | Decided by the run (either way is recorded) |
| P5 | Throughput at page-1 depth, one Python client with threads: the engine holds a **p99 under 100 ms at 16 concurrent requests** (GPU). This client is probably the limit above a few hundred QPS, for both engines. | baseline/README.md (275 QPS under 100 ms, a different pipeline) | Engine p99 over 100 ms at 16 concurrent |
| P6 | Indexing time: **unsourced**. Belief 0.5 that the engine's `PUT` (Parquet on disk, GPU or CPU build) is faster than OpenSearch's bulk upload plus force-merge. | unsourced | Decided by the run |

These are for the **default depth** (page 1, 10 hits; quality uses 100). They say nothing about depth 2,000, which is what `colab/fair_bench.py` loads.

**What is and is not matched.** Matched: the same catalogue (rows and columns, the synthetic ones too), the same query texts, HTTP on localhost with keep-alive from the same Python client,
the same page depth, one single-node server of each, one segment each (OpenSearch is force-merged), batch-1 requests one at a time, the engine's cache off.
**Not matched:** each engine uses its own tuned lexical request (`bm25f-tuned.json` scored as BM25F with per-field `b`, `k1` 0.036, the engine's `s3_english` analyzer; against
`opensearch-cross-fields-tuned.json` with OpenSearch's built-in `english` analyzer and ONE `k1`/`b` for the index, 1.2/0.75, as the prototype); the hardware (the engine on the GPU plus CPU, OpenSearch on CPU only, both servers
alive on one VM, unpinned); the indexing path (the engine reads a Parquet file; OpenSearch is fed JSON bulk requests by `referee/opensearch.py`); the engine runs as one process with the standard-library front
(not the multi-process GPU layout of `fair_bench`); the load is a closed loop of Python threads, so it **hides queueing** (coordinated omission), unlike `fair_bench`'s open loop.

In [ ]:
# @title OpenSearch: download, start, index the same catalogue
OS_UP, OS_SKIP_REASON, OS_BASE, OS_INDEX = False, None, None, None
OS_INDEX_S = OS_MERGE_S = None

def mem_gb():
    try:
        return int(next(l for l in open("/proc/meminfo") if l.startswith("MemTotal")).split()[1]) / 1048576
    except Exception:
        return 8.0

def start_opensearch():
    # Adapted from colab/fair_bench.py start_opensearch: the same tarball, single node, security off, run as an
    # unprivileged user when we are root (OpenSearch refuses to run as root). No CPU pinning here.
    home = WORK / f"opensearch-{OS_VERSION}"
    if not (home / "bin" / "opensearch").exists():
        tgz = WORK / f"opensearch-{OS_VERSION}.tar.gz"
        if not tgz.exists():
            say(f"downloading OpenSearch {OS_VERSION} (about 1 GB)")
            urllib.request.urlretrieve(f"https://artifacts.opensearch.org/releases/bundle/opensearch/{OS_VERSION}/"
                                       f"opensearch-{OS_VERSION}-linux-x64.tar.gz", tgz)
        with tarfile.open(tgz) as t:
            t.extractall(WORK)
    heap = f"{max(1, int(min(OS_HEAP_MAX_GB, 0.4 * mem_gb())))}g"
    port = free_port()
    prefix = []
    if os.geteuid() == 0:
        subprocess.run(["useradd", "-m", "osuser"], stderr=subprocess.DEVNULL)
        subprocess.run(["chown", "-R", "osuser:osuser", str(home)], check=True)
        prefix = ["runuser", "-u", "osuser", "--", "env"]
    knn_lib = home / "plugins" / "opensearch-knn" / "lib"
    cmd = prefix + [f"OPENSEARCH_JAVA_OPTS=-Xms{heap} -Xmx{heap}", f"LD_LIBRARY_PATH={knn_lib}", str(home / "bin" / "opensearch"),
                    "-Ediscovery.type=single-node", "-Eplugins.security.disabled=true", f"-Ehttp.port={port}",
                    f"-Epath.data={WORK / 'os-data'}", f"-Epath.logs={WORK / 'os-logs'}"]
    log = WORK / "opensearch.log"
    base = f"http://127.0.0.1:{port}"
    (WORK / "os-data").mkdir(exist_ok=True); (WORK / "os-logs").mkdir(exist_ok=True)
    if os.geteuid() == 0:
        subprocess.run(["chown", "-R", "osuser:osuser", str(WORK / "os-data"), str(WORK / "os-logs")], check=True)
    proc = spawn("opensearch", cmd, log)
    wait_until(lambda: http("GET", base)["version"]["number"], "opensearch", 420, proc, log)
    say(f"opensearch {http('GET', base)['version']['number']} is up at {base}, heap {heap}")
    return base

if RUN_OPENSEARCH:
    try:
        OS_BASE = start_opensearch()
        os.environ["S3_SEARCH_OS_ANALYZER"] = "english"          # the prototype indexed OpenSearch with `english`
        os.environ["S3_SEARCH_OPENSEARCH_URL"] = OS_BASE         # referee/opensearch.py reads it when imported
        from referee import opensearch as osref
        os_mapping = copy.deepcopy(ds.mapping)                   # scalar k1/b: OpenSearch has one similarity per index
        os_mapping["properties"].update(EXTRA_PROPS)
        os_mapping["_meta"]["engine"]["hot"] = ["price", "stock"]
        t0 = time.time()
        say(f"OpenSearch: indexing the same {len(cat):,} products (referee/opensearch.py: bulk of 5,000, then refresh)")
        os_ix = osref.build_index(cat, os_mapping)
        OS_INDEX = os_ix.name
        OS_INDEX_S = time.time() - t0
        t0 = time.time()
        http("POST", f"{OS_BASE}/{OS_INDEX}/_forcemerge?max_num_segments=1", timeout=7200)
        OS_MERGE_S = time.time() - t0
        n = http("GET", f"{OS_BASE}/{OS_INDEX}/_count")["count"]
        assert n == len(cat), f"OpenSearch holds {n} documents, expected {len(cat)}"
        OS_UP = True
        say(f"OpenSearch: {n:,} docs; bulk + refresh {OS_INDEX_S:.1f} s, force-merge to one segment {OS_MERGE_S:.1f} s")
    except Exception as ex:
        OS_SKIP_REASON = f"{type(ex).__name__}: {ex}"
        print("OPENSEARCH SKIPPED, the benchmark below runs for the engine only.\nReason:", OS_SKIP_REASON[:1500])
        if (WORK / "opensearch.log").exists():
            print("opensearch.log tail:\n" + tail(WORK / "opensearch.log", 25))
        stop_proc(PROCS.pop("opensearch", None))
else:
    OS_SKIP_REASON = "RUN_OPENSEARCH is False"
    print("OpenSearch skipped:", OS_SKIP_REASON)

In [ ]:
# @title The two targets: the same query text becomes each engine's own tuned request
class Client:
    # A POST client with one keep-alive connection per thread (the same for both engines).
    def __init__(self, url):
        u = urllib.parse.urlparse(url)
        self.host, self.port, self.path, self.local = u.hostname, u.port, u.path, threading.local()
    def post(self, body, timeout=60):
        data = body if isinstance(body, bytes) else json.dumps(body).encode()
        for attempt in (0, 1):
            c = getattr(self.local, "c", None)
            if c is None:
                c = self.local.c = HTTPConnection(self.host, self.port, timeout=timeout)
            try:
                c.request("POST", self.path, data, {"Content-Type": "application/json"})
                r = c.getresponse()
                return r.status, r.read()
            except (HTTPException, OSError):
                c.close(); self.local.c = None
                if attempt:
                    raise

OS_TMPL = json.loads((REPO / "referee/requests/opensearch-cross-fields-tuned.json").read_text())
def os_body(q, size=DEPTH):
    b = copy.deepcopy(OS_TMPL)
    b["query"]["multi_match"]["query"] = q
    return {**b, "size": size, "_source": False, "track_total_hits": False}

TARGETS = {"engine": {"label": "engine (BM25F tuned)", "client": Client(f"{ENGINE}/indexes/{INDEX}/_search"),
                      "body": lambda q, size=DEPTH: engine_body(q, size), "request": ENGINE_TMPL}}
if OS_UP:
    TARGETS["opensearch"] = {"label": "OpenSearch 2.19.1 (cross_fields tuned)", "client": Client(f"{OS_BASE}/{OS_INDEX}/_search"),
                             "body": os_body, "request": OS_TMPL}
for name, t in TARGETS.items():
    st, raw = t["client"].post(t["body"](DEMO_QUERIES[0], 3))
    top = [h["_id"] for h in json.loads(raw)["hits"]["hits"]]
    print(f"{name:<11} HTTP {st}; top 3 for {DEMO_QUERIES[0]!r}: {top}")
print("targets:", list(TARGETS), "" if OS_UP else f"(OpenSearch absent: {OS_SKIP_REASON})")

### Quality: nDCG@10 and exact recall@100, fit split only

`N_QUALITY` queries drawn with a fixed seed from the **fit** (train) split, scored with `referee.metrics` against the **fit** judgments (`loader.fit_judgments`).
The test judgments are never loaded here: they belong to the referee's scorer. An unjudged product counts as gain 0 (the referee's rule), so these are not comparable
with a test-split score of record.

In [ ]:
# @title Quality
rng = np.random.default_rng(0)
fit_ids = np.sort(ds.fit["query_id"].unique())
chosen = [int(x) for x in rng.choice(fit_ids, size=min(N_QUALITY, len(fit_ids)), replace=False)]
text_of = ds.fit.drop_duplicates("query_id").set_index("query_id")["query"].to_dict()
judgments = {q: j for q, j in loader.fit_judgments(ds).items() if q in set(chosen)}

def ranked(target, qid, size=100):
    st, raw = target["client"].post(target["body"](text_of[qid], size))
    if st != 200:
        raise RuntimeError(f"HTTP {st}: {raw[:300]!r}")
    return qid, [h["_id"] for h in json.loads(raw)["hits"]["hits"]]

QUALITY = {}
for name, t in TARGETS.items():
    t0 = time.time()
    with ThreadPoolExecutor(8) as ex:
        runs = dict(ex.map(lambda q: ranked(t, q), chosen))
    s = metrics.score_run(runs, judgments, 10, (100,))["mean"]
    QUALITY[name] = {"ndcg@10": s["ndcg@10"], "exact_recall@100": s["exact_recall@100"], "n_queries": len(chosen),
                     "seconds": round(time.time() - t0, 1)}
    say(f"{name:<11} nDCG@10 {s['ndcg@10']:.4f}   exact recall@100 {s['exact_recall@100']:.4f}   ({len(chosen)} fit queries, {time.time() - t0:.1f} s)")

### Latency: batch 1, one request at a time

`N_LATENCY` test **query texts** (no judgments are involved), page 1 (`DEPTH` hits, ids and scores), sequentially, after a warm-up of 30 fit queries. Time is the client's wall clock around
the whole HTTP call.

In [ ]:
# @title Latency, batch 1
lat_texts = [str(t) for t in ds.test_queries["query"].drop_duplicates()][:N_LATENCY]
warm_texts = [text_of[q] for q in chosen[:30]]
pct = lambda a, p: float(np.percentile(a, p))

def batch1(target, texts):
    ms, errors = [], 0
    for q in texts:
        body = target["body"](q)
        t0 = time.perf_counter()
        st, _ = target["client"].post(body)
        dt = (time.perf_counter() - t0) * 1000
        if st == 200: ms.append(dt)
        else: errors += 1
    return ms, errors

LATENCY, LAT_SAMPLES = {}, {}
for name, t in TARGETS.items():
    batch1(t, warm_texts)
    ms, errors = batch1(t, lat_texts)
    LAT_SAMPLES[name] = ms
    LATENCY[name] = {"n": len(ms), "errors": errors, "p50_ms": pct(ms, 50), "p95_ms": pct(ms, 95), "p99_ms": pct(ms, 99),
                     "mean_ms": float(np.mean(ms)), "max_ms": max(ms)}
    L = LATENCY[name]
    say(f"{name:<11} p50 {L['p50_ms']:.1f}  p95 {L['p95_ms']:.1f}  p99 {L['p99_ms']:.1f} ms  ({L['n']} queries, {errors} errors)")

### Throughput: a short closed loop

For each concurrency level, that many Python threads each send requests back to back for `SECONDS_PER_LEVEL` seconds (different query texts), one engine at a time.
This is **deliberately modest**: a closed loop slows its own clock when the server slows, and one Python process caps out at a few hundred to a thousand or so requests a second, so a high-concurrency
QPS here can be the client's limit and not the server's. `colab/fair_bench.py` is the rigorous version.

In [ ]:
# @title Throughput
def load_level(target, texts, conc, seconds):
    bodies = [target["body"](q) for q in texts]
    deadline = time.perf_counter() + seconds
    def worker(i):
        j, lat, errs = (i * 37) % len(bodies), [], 0
        while time.perf_counter() < deadline:
            t0 = time.perf_counter()
            try:
                st, _ = target["client"].post(bodies[j % len(bodies)])
            except Exception:
                st = 0
            dt = (time.perf_counter() - t0) * 1000
            if st == 200: lat.append(dt)
            else: errs += 1
            j += 1
        return lat, errs
    t0 = time.perf_counter()
    with ThreadPoolExecutor(conc) as ex:
        res = list(ex.map(worker, range(conc)))
    elapsed = time.perf_counter() - t0
    lat = [x for r in res for x in r[0]]
    return {"concurrency": conc, "seconds": round(elapsed, 1), "requests": len(lat), "errors": sum(r[1] for r in res),
            "qps": len(lat) / elapsed, "p50_ms": pct(lat, 50) if lat else None, "p99_ms": pct(lat, 99) if lat else None}

THROUGHPUT = {name: [] for name in TARGETS}
for name, t in TARGETS.items():
    for conc in CONCURRENCY:
        r = load_level(t, lat_texts + warm_texts, conc, SECONDS_PER_LEVEL)
        THROUGHPUT[name].append(r)
        say(f"{name:<11} {conc:>3} threads: {r['qps']:7.0f} QPS   p50 {r['p50_ms']:.1f}  p99 {r['p99_ms']:.1f} ms   errors {r['errors']}")

In [ ]:
# @title Results table, chart, and demo-bench.json
print("DEVELOPMENT ONLY: not the score of record (that is referee/colab_referee.ipynb).\n")
rows = {"indexing, seconds": {"engine": ENGINE_INDEX_S, "opensearch": OS_INDEX_S},
        "  (OpenSearch force-merge, seconds)": {"opensearch": OS_MERGE_S}}
for name in TARGETS:
    q, l = QUALITY[name], LATENCY[name]
    rows.setdefault("nDCG@10 (fit split)", {})[name] = q["ndcg@10"]
    rows.setdefault("exact recall@100 (fit split)", {})[name] = q["exact_recall@100"]
    for k, lab in (("p50_ms", "batch-1 p50, ms"), ("p95_ms", "batch-1 p95, ms"), ("p99_ms", "batch-1 p99, ms")):
        rows.setdefault(lab, {})[name] = l[k]
    for r in THROUGHPUT[name]:
        rows.setdefault(f"{r['concurrency']} threads: QPS", {})[name] = r["qps"]
        rows.setdefault(f"{r['concurrency']} threads: p99, ms", {})[name] = r["p99_ms"]
table = pd.DataFrame(rows).T.reindex(columns=list(TARGETS))
table.columns = [TARGETS[c]["label"] for c in table.columns]
display_table = table.map(lambda v: "" if v is None or (isinstance(v, float) and math.isnan(v)) else (f"{v:.4f}" if abs(v) < 1 else f"{v:,.1f}"))
print(display_table.to_string())
print(f"\n{DATASET}: {len(cat):,} products; device {DEVICE}" + (f" ({GPU_NAME})" if GPU_NAME else "")
      + f"; commit {COMMIT}; depth {DEPTH}; {len(lat_texts)} latency queries; {len(chosen)} quality queries"
      + ("" if OS_UP else f"\nOPENSEARCH NOT RUN: {OS_SKIP_REASON}"))

import matplotlib
if not IN_COLAB:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
COLORS = {"engine": "#0072B2", "opensearch": "#D55E00"}          # colour-blind safe pair, fixed per engine
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for name in TARGETS:
    ms = np.sort(LAT_SAMPLES[name])
    a1.plot(ms, np.arange(1, len(ms) + 1) / len(ms), color=COLORS[name], lw=2, label=TARGETS[name]["label"])
    a2.plot([r["qps"] for r in THROUGHPUT[name]], [r["p99_ms"] for r in THROUGHPUT[name]], "o-", color=COLORS[name], lw=2,
            label=TARGETS[name]["label"])
    for r in THROUGHPUT[name]:
        a2.annotate(str(r["concurrency"]), (r["qps"], r["p99_ms"]), textcoords="offset points", xytext=(4, 4), fontsize=8, color="#444")
a1.set_xscale("log"); a1.set_xlabel("batch-1 latency, ms (log scale)"); a1.set_ylabel("fraction of requests at or under"); a1.set_title("Latency distribution, one request at a time")
a2.set_xlabel("achieved QPS"); a2.set_ylabel("p99 latency, ms"); a2.set_yscale("log"); a2.set_title("p99 against QPS (labels: concurrent threads)")
from matplotlib.ticker import FuncFormatter
plain = FuncFormatter(lambda v, _: f"{v:g}")
for a in (a1, a2):
    a.grid(alpha=0.25); a.spines[["top", "right"]].set_visible(False); a.legend(frameon=False, loc="best")
a1.xaxis.set_major_formatter(plain); a1.xaxis.set_minor_formatter(plain)
a2.yaxis.set_major_formatter(plain); a2.yaxis.set_minor_formatter(plain)
fig.suptitle(f"DEVELOPMENT ONLY, never a score of record  ({DATASET}, {DEVICE})", fontsize=9, color="#a00")
CHART = WORK / "demo-bench.png"
fig.savefig(CHART, dpi=130)
plt.show()

OUT_JSON = Path("/content/demo-bench.json") if IN_COLAB else WORK / "demo-bench.json"
out = {"kind": "demo-bench", "label": "development only, never a score of record", "ref": REF if IN_COLAB else "local checkout",
       "commit": COMMIT, "dataset": DATASET, "n_products": int(len(cat)), "device": DEVICE, "gpu": GPU_NAME, "cpus": os.cpu_count(),
       "opensearch_version": OS_VERSION if OS_UP else None, "opensearch_skipped": OS_SKIP_REASON,
       "settings": {"depth": DEPTH, "n_quality": len(chosen), "n_latency": len(lat_texts), "concurrency": CONCURRENCY,
                    "seconds_per_level": SECONDS_PER_LEVEL, "quality_split": "fit (train) judgments only", "client": "python threads, keep-alive, closed loop"},
       "indexing": {"engine_put_s": ENGINE_INDEX_S, "opensearch_bulk_refresh_s": OS_INDEX_S, "opensearch_forcemerge_s": OS_MERGE_S},
       "requests": {n: t["request"] for n, t in TARGETS.items()},
       "quality": QUALITY, "latency_batch1": LATENCY, "throughput": THROUGHPUT}
OUT_JSON.write_text(json.dumps(out, indent=1, default=float))
say(f"wrote {OUT_JSON} and {CHART}")

### Judge it against the predictions

Open the table above next to the predictions cell and mark each of P1 to P6 as held or wrong, **wrong ones included**, in the floor's `company/milestones.json`
(the lab repository). Reminder of what this run is: one machine, one client, seconds per level. It is a development check, never a score.

## Stop everything

Kills the serving process and OpenSearch (both are started in their own process groups, so a kernel restart would not stop them).

In [ ]:
stop_all()
print("servers stopped; files are in", WORK)